# [STARTER] Udaplay Project

## Part 01 - Offline RAG

In this part of the project, you'll build your VectorDB using Chroma.

The data is inside folder `project/starter/games`. Each file will become a document in the collection you'll create.
Example.:
```json
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}
```


### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import os
import json
import time
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv


In [3]:
# TODO: Create a .env file with the following variables
# OPENAI_API_KEY="YOUR_KEY"
# CHROMA_OPENAI_API_KEY="YOUR_KEY"
# TAVILY_API_KEY="YOUR_KEY"

ENV_PATH = ".env"
CONFIG_PATH = "config.env"


In [4]:
# TODO: Load environment variables
# load_dotenv()

load_dotenv(ENV_PATH)
load_dotenv(CONFIG_PATH)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
CHROMA_OPENAI_API_KEY = os.getenv("CHROMA_OPENAI_API_KEY") or OPENAI_API_KEY
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL", "https://openai.vocareum.com/v1")
MAX_RETRIES = 10

os.environ["OPENAI_BASE_URL"] = OPENAI_BASE_URL
assert OPENAI_API_KEY, "OPENAI_API_KEY is required"
assert CHROMA_OPENAI_API_KEY, "CHROMA_OPENAI_API_KEY or OPENAI_API_KEY is required"


### VectorDB Instance

In [5]:
# TODO: Instantiate your ChromaDB Client
# Choose any path you want
# chroma_client = chromadb.PersistentClient(path="chromadb")

CHROMA_PATH = "chromadb"
COLLECTION_NAME = "udaplay"
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)


### Collection

In [6]:
# TODO: Pick one embedding function
# If picking something different than openai, 
# make sure you use the same when loading it
# embedding_fn = embedding_functions.OpenAIEmbeddingFunction()

embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=CHROMA_OPENAI_API_KEY,
    api_base=OPENAI_BASE_URL,
    model_name="text-embedding-3-small",
)
embedding_fn.client = embedding_fn.client.with_options(max_retries=MAX_RETRIES)


In [7]:
# TODO: Create a collection
# Choose any name you want
# collection = chroma_client.create_collection(
#    name="udaplay",
#    embedding_function=embedding_fn
#)

collection = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME,
    embedding_function=embedding_fn,
)

def retry_call(fn, max_retries=MAX_RETRIES, delay=1):
    for attempt in range(1, max_retries + 1):
        try:
            return fn()
        except Exception:
            if attempt == max_retries:
                raise
            time.sleep(delay)
            delay = min(delay * 2, 20)


### Add documents

In [8]:
# Make sure you have a directory "project/starter/games"
data_dir = "games"
indexed = 0

for file_name in sorted(os.listdir(data_dir)):
    if not file_name.endswith(".json"):
        continue

    file_path = os.path.join(data_dir, file_name)
    with open(file_path, "r", encoding="utf-8") as f:
        game = json.load(f)

    # Index text
    content = (
        f"Name: {game['Name']}\n"
        f"Platform: {game['Platform']}\n"
        f"Genre: {game['Genre']}\n"
        f"Publisher: {game['Publisher']}\n"
        f"YearOfRelease: {game['YearOfRelease']}\n"
        f"Description: {game['Description']}"
    )

    metadata = {
        "Name": game["Name"],
        "Platform": game["Platform"],
        "Genre": game["Genre"],
        "Publisher": game["Publisher"],
        "YearOfRelease": game["YearOfRelease"],
        "Description": game["Description"],
    }

    retry_call(lambda: collection.upsert(
        ids=[file_name.replace(".json", "")],
        documents=[content],
        metadatas=[metadata],
    ))
    indexed += 1

print(f"Indexed {indexed} games into '{COLLECTION_NAME}' at '{CHROMA_PATH}'.")
print(retry_call(lambda: collection.query(query_texts=["first 3D Mario platformer"], n_results=3)))

Indexed 15 games into 'udaplay' at 'chromadb'.


{'ids': [['009', '008', '012']], 'embeddings': None, 'documents': [["Name: Super Mario 64\nPlatform: Nintendo 64\nGenre: Platformer\nPublisher: Nintendo\nYearOfRelease: 1996\nDescription: A groundbreaking 3D platformer that set new standards for the genre, featuring Mario's quest to rescue Princess Peach.", 'Name: Super Mario World\nPlatform: Super Nintendo Entertainment System (SNES)\nGenre: Platformer\nPublisher: Nintendo\nYearOfRelease: 1990\nDescription: A classic platformer where Mario embarks on a quest to save Princess Toadstool and Dinosaur Land from Bowser.', 'Name: Mario Kart 8 Deluxe\nPlatform: Nintendo Switch\nGenre: Racing\nPublisher: Nintendo\nYearOfRelease: 2017\nDescription: An enhanced version of Mario Kart 8, featuring new characters, tracks, and improved gameplay mechanics.']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'Genre': 'Platformer', 'Publisher': 'Nintendo', 'Platform': 'Nintendo 64', 'Name': 'Super Mario